# 4 · Policy as documents, loaded on demand

Typologies, risk appetite and escalation thresholds belong in files an analyst
can edit without touching code. With policy in Python, moving a threshold is a
pull request. With policy in files, it is somebody editing a document.

| Level | What | In the prompt |
|---|---|---|
| 1 | name + description | always, because it is tiny |
| 2 | the full document body | only after an agent asks |

In [ ]:
# Reload the package from disk on every run, so an edit to src/sentinel takes
# effect without restarting the kernel. Python caches imported modules in
# sys.modules and a stale one will happily report yesterday's numbers.
import sys, pathlib
for name in [m for m in sys.modules if m.startswith("sentinel")]:
    del sys.modules[name]

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("sentinel package:", ROOT / "src" / "sentinel")

In [ ]:
from sentinel import middleware

print(middleware.policy_catalog())
print()
print(middleware.disclosure_stats())

## The catalog is re-scanned on every model call

Not cached at import. That is what makes an analyst's edit take effect on the
next call rather than after a restart.

In [ ]:
from sentinel.config import POLICIES_DIR

before = {p["name"] for p in middleware.discover_policies()}

probe = POLICIES_DIR / "_scratch.md"
probe.write_text("---\nname: _scratch\ndescription: written while this kernel was running.\n---\n\nIf you can read this, the catalog was re-scanned.\n", encoding="utf-8")

after = {p["name"] for p in middleware.discover_policies()}
print("newly visible:", sorted(after - before) or "NOTHING - it was cached")

probe.unlink()
print("removed; back to", len(middleware.discover_policies()), "documents")

## Loading is a rule, not a request

A prompt asking an agent to read the policy first is advisory — the model mostly
complies. `PolicyGateMiddleware` short-circuits `wrap_tool_call` and returns an
error **instead of** running the tool, so a model that ignores the instruction
still cannot file a verdict.

In [ ]:
for tool_name, needed in middleware.POLICY_GATES.items():
    print(f"{tool_name:<20} blocked until  {needed}")

In [ ]:
# The gate in action, on a bare agent with no policy loaded.
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from sentinel.tools.disposition_tools import DISPOSITION_TOOLS
from sentinel.agents.disposition import PROMPT as DISPOSITION_PROMPT
from sentinel import db

db.init_runtime()
gated = create_agent(
    init_chat_model("gpt-4.1-mini", model_provider="openai"),
    tools=DISPOSITION_TOOLS,
    system_prompt=DISPOSITION_PROMPT,
    middleware=[middleware.PolicyGateMiddleware(middleware.POLICY_GATES)],
    state_schema=middleware.PolicyState,
)
out = gated.invoke({"messages": [{"role": "user", "content":
    "Record A00985 as legitimate, high confidence, citing note N00080. Do not load any policy."}],
    "account_id": "A00985"})
for m in out["messages"]:
    if m.type == "tool" and "BLOCKED" in (m.text or ""):
        print(m.text)

## Every load is recorded

So that "loaded on demand" is provable after a run rather than asserted.

In [ ]:
rows = db.fetch("SELECT policy, COUNT(*) n FROM policy_loads GROUP BY policy ORDER BY n DESC")
for r in rows:
    print(f"{r['policy']:<22} {r['n']}")
if not rows:
    print("(no loads recorded yet - run notebook 03 or a sweep first)")